### Import Libraries

In [1]:
import os
import shutil
from huggingface_hub import hf_hub_download
import zipfile
import pandas as pd
from pathlib import Path
import ast

/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialization

In [2]:
repo_id = "AI4Patents/IMPACT"

dataset_dir = Path("impact_dataset")

years = range(2007, 2023 + 1)

### Download Dataset

In [8]:
os.makedirs(dataset_dir, exist_ok=True)

In [9]:
def download_file(filename: str):
    print(f"Downloading {filename} ...")

    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        local_dir=dataset_dir,
        local_dir_use_symlinks=False,
        resume_download=True
    )

    print(f"Saved to {path}")

In [10]:
for year in years:
    csv_name = f"{year}.csv"
    zip_name = f"{year}.zip"

    download_file(csv_name)
    download_file(zip_name)

Saved to impact_dataset/2007.csv
Saved to impact_dataset/2007.zip
Saved to impact_dataset/2008.csv
Saved to impact_dataset/2008.zip
Saved to impact_dataset/2009.csv
Saved to impact_dataset/2009.zip
Saved to impact_dataset/2010.csv
Saved to impact_dataset/2010.zip
Saved to impact_dataset/2011.csv
Saved to impact_dataset/2011.zip
Saved to impact_dataset/2012.csv
Saved to impact_dataset/2012.zip
Saved to impact_dataset/2013.csv
Saved to impact_dataset/2013.zip
Saved to impact_dataset/2014.csv
Saved to impact_dataset/2014.zip
Saved to impact_dataset/2015.csv
Saved to impact_dataset/2015.zip
Saved to impact_dataset/2016.csv
Saved to impact_dataset/2016.zip
Saved to impact_dataset/2017.csv
Saved to impact_dataset/2017.zip
Saved to impact_dataset/2018.csv
Saved to impact_dataset/2018.zip
Saved to impact_dataset/2019.csv
Saved to impact_dataset/2019.zip
Saved to impact_dataset/2020.csv
Saved to impact_dataset/2020.zip
Saved to impact_dataset/2021.csv
Saved to impact_dataset/2021.zip
Saved to i

EntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a3b0970-354c9c9a117c5f2179760d1a;017d95a3-75c5-484f-ab47-df8cc1d4637a)

Entry Not Found for url: https://huggingface.co/datasets/AI4Patents/IMPACT/resolve/main/2023.csv.

### Unzip Dataset

In [3]:
def delete_zip_by_name(zip_name: str, data_dir=dataset_dir):
    data_dir = Path(data_dir)

    target = zip_name if zip_name.lower().endswith(".zip") else f"{zip_name}.zip"
    zip_path = data_dir / target

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip not found: {zip_path}")

    zip_path.unlink()
    print(f"Deleted: {zip_path.name}")
    return


In [4]:
def remove_macosx(folder_name: str, data_dir: Path):
    target_dir = data_dir / folder_name
    temp_dir = data_dir / "temp"

    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)

    kept = None

    for child in target_dir.iterdir():
        if child.is_dir() and child.name == folder_name:
            kept = child
        else:
            shutil.rmtree(child) if child.is_dir() else child.unlink()

    if kept is None:
        raise FileNotFoundError(f"Did not find '{folder_name}' inside {target_dir}")

    moved = temp_dir / kept.name
    kept.rename(moved)

    shutil.rmtree(target_dir)
    moved.rename(target_dir)
    shutil.rmtree(temp_dir)

In [5]:
def unzip_all(data_dir=dataset_dir):
    zip_files = [f for f in os.listdir(data_dir) if f.endswith(".zip")]

    if not zip_files:
        print("No zip files found.")
        return

    for zfile in zip_files:
        zip_path = os.path.join(data_dir, zfile)

        if not os.path.isfile(zip_path):
            print(zip_path)
            print(f"[ERROR] File not found, skipping: {zip_path}")
            continue

        folder_name = zfile.replace(".zip", "")
        extract_dir = os.path.join(data_dir, folder_name)

        if os.path.exists(extract_dir):
            print(f"[SKIPPED] {zfile} already extracted.")
            continue

        print(f"[UNZIPPING] {zfile} → {extract_dir}")

        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(f"[DONE] Extracted to: {extract_dir}")

        remove_macosx(folder_name, dataset_dir)
        delete_zip_by_name(folder_name, dataset_dir)


    print("\nAll .zip files processed.")

In [6]:
unzip_all()

[UNZIPPING] 2014.zip → impact_dataset/2014
[DONE] Extracted to: impact_dataset/2014
Deleted: 2014.zip
[UNZIPPING] 2021.zip → impact_dataset/2021
[DONE] Extracted to: impact_dataset/2021
Deleted: 2021.zip
[UNZIPPING] 2013.zip → impact_dataset/2013
[DONE] Extracted to: impact_dataset/2013
Deleted: 2013.zip
[UNZIPPING] 2015.zip → impact_dataset/2015
[DONE] Extracted to: impact_dataset/2015
Deleted: 2015.zip
[UNZIPPING] 2019.zip → impact_dataset/2019
[DONE] Extracted to: impact_dataset/2019
Deleted: 2019.zip
[UNZIPPING] 2018.zip → impact_dataset/2018
[DONE] Extracted to: impact_dataset/2018
Deleted: 2018.zip
[UNZIPPING] 2007.zip → impact_dataset/2007
[DONE] Extracted to: impact_dataset/2007
Deleted: 2007.zip
[UNZIPPING] 2011.zip → impact_dataset/2011
[DONE] Extracted to: impact_dataset/2011
Deleted: 2011.zip
[UNZIPPING] 2008.zip → impact_dataset/2008
[DONE] Extracted to: impact_dataset/2008
Deleted: 2008.zip
[UNZIPPING] 2020.zip → impact_dataset/2020
[DONE] Extracted to: impact_dataset/202

## Dataframe Management

In [7]:
list_cols = ["class_search", "file_names", "fig_desc"]
text_cols = ["title", "claim", "caption"]

In [8]:
def find_img_path(sample, data_dir=dataset_dir):
    year = sample["year"]
    sample_file_names = sample["file_names"]

    if not isinstance(sample_file_names, (list, tuple)) or len(sample_file_names) == 0:
        return []

    first_name = sample_file_names[0]

    if not isinstance(first_name, str):
        return []

    parts = first_name.split("-")
    if len(parts) >= 2:
        folder_name = "-".join(parts[:2])
    else:
        folder_name = parts[0]

    # 4. Build folder path (cast year to str for safety)
    folder_path = os.path.join(data_dir, str(year), folder_name)

    img_paths = [
        os.path.join(folder_path, fn)
        for fn in sample_file_names
        if isinstance(fn, str)
    ]

    return img_paths


In [9]:
def safe_literal_eval(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if not isinstance(x, str) or x.strip() == "":
        return []
    try:
        v = ast.literal_eval(x)
    except Exception:
        return []
    
    return list(v) if isinstance(v, (list, tuple)) else []

In [10]:
def merge_all_csv(
    data_dir=dataset_dir, 
    source_file="year", 
    list_cols=None,
    year_list: list = []
):
    csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

    if year_list == []:
        filtered_csv = csv_files
    else:
        filtered_csv = [f for f in csv_files if f.replace(".csv", "") in year_list]

    if not filtered_csv:
        print("No csv files found.")
        return

    frames = []
    for csv in filtered_csv:
        csv_path = os.path.join(data_dir, csv)
        df = pd.read_csv(csv_path)

        for col in list_cols:
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: safe_literal_eval(x)
                    if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
                    else x
                )
        
        filename = os.path.basename(csv_path).split('.')[0]

        df[source_file] = filename
        cols = [source_file] + [c for c in df.columns if c != source_file]
        df = df[cols]

        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)

    merged["file_names"] = merged.apply(
        lambda row: find_img_path(row, data_dir=data_dir),
        axis=1,
    )
    
    return merged

        

In [11]:
Impact_df = merge_all_csv(list_cols=list_cols)

In [12]:
cols = ["title", "caption", "file_names", "fig_desc", "class"]

Impact_df = Impact_df[cols]

In [13]:
Impact_df.head()

,title,caption,file_names,fig_desc,class
0,Combination card and key holder,The image is a white outline of a combination...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,D 3208
1,Container,"The image is a white drawing of a container, ...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...","D 9542, D9574"
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,D14486
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,D14486
4,Bottle cap opener,The image is a drawing of a bottle cap opener...,[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,D 8 40


In [14]:
Impact_df.shape[0]

434498

### Save Impact DF

In [15]:
Impact_df.to_csv("model_io/Base.csv", index=False, encoding="utf-8")

### Load Impact DF

In [1]:
import ast
import pandas as pd

In [2]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [ ]:
Impact_df = pd.read_csv("Impact.csv",encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,file_names,Loc_class,main_class,sub_class,llava_output,components
0,Article of footwear,[impact_dataset/2022/USD0943877-20220222/USD09...,{02-04},2,4,The image you've provided appears to be a tech...,"sole, heel, toe, upper, tongue, laces, eyelets"
1,Shoe,[impact_dataset/2022/USD0950931-20220510/USD09...,{02-04},2,4,"The image shows a pair of shoes, and I will de...","sole, heel, upper, tongue, laces, eyelets"
2,Rear combination lamp for automobile,[impact_dataset/2022/USD0957706-20220712/USD09...,"{02-07, 26-06}",2,7,The image you've provided appears to be a gray...,"base, housing, lens, switch, cord, bulb, refle..."
3,Portable light beacon,[impact_dataset/2022/USD0959716-20220802/USD09...,"{02-07, 26-02}",2,7,The image you've provided appears to be a tech...,"base, stem, head, lens, switch, battery compar..."
4,Water shoe,[impact_dataset/2022/USD0960535-20220816/USD09...,{02-04},2,4,The image you've provided appears to be a line...,"sole, upper, tongue, laces, heel, toe cap"
